### Path Finder: What are the potential paths between two nodes?
### This pipeline can be used to get a ranked path between A and B given a set of paths.

## Quick start: `pathfinder` API

TCT now provides a developer-friendly `pathfinder` wrapper as part of the main API. It resolves labels/CURIEs, loads and caches Translator resources, selects compatible APIs/predicates, queries KPs in parallel, and returns a `FinderResult` with parsed graph sections surfaced directly. Use the fine-grained workflow below when you need endpoint selection, predicate control, raw query construction, or visualization setup.


In [1]:
from TCT import get_translator_resources, query_TCT_pathfinder

resources = get_translator_resources()
paths = query_TCT_pathfinder(
    start="asthma",
    end="albuterol",
    intermediate_categories=["Gene", "Protein"],
    resources=resources,
)
paths.resolved_nodes


Skipping server without x-maturity: {'url': '/sipr'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
59
(18613, 5)
(30172, 5)
Microbiome KP - TRAPI 1.5.0: Success!
CATRAX Pharmacogenomics KP - TRAPI 1.5.0: Success!
Genetics Data Provider for NCATS Biomedical Translator Reasoners: Success!
RTX KG2 - TRAPI 1.5.0: Success!
Automat-robokop(Trapi v1.5.0): Success!
Service Provider TRAPI: Success!
BioThings Explorer (BTE) TRAPI: Success!
CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0: Success!
CATRAX Pharmacogenomics KP - TRAPI 1.5.0: Success!
RTX KG2 - TRAPI 1.5.0: Success!


{'start': ResolvedNode(input_value='asthma', curie='MONDO:0004979', label='asthma', categories=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing']),
 'end': ResolvedNode(input_value='albuterol', curie='CHEBI:2549', label='Salbutamol', categories=['biolink:SmallMolecule', 'biolink:MolecularEntity', 'biolink:ChemicalEntity', 'biolink:PhysicalEssence', 'biolink:ChemicalOrDrugOrTreatment', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:ChemicalEntityOrProteinOrPolypeptide', 'biolink:NamedThing', 'biolink:PhysicalEssenceOrOccurrent'])}

In [ ]:
paths.resolved_nodes['start'].input_value

ResolvedNode(input_value='asthma', curie='MONDO:0004979', label='asthma', categories=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing'])

In [3]:
{
    "node_count": len(paths.knowledge_graph.get("nodes", {})),
    "edge_count": len(paths.knowledge_graph.get("edges", {})),
    "result_count": len(paths.results),
    "raw_sections": list(paths.to_dict().keys()),
}


{'node_count': 6,
 'edge_count': 30,
 'result_count': 1,
 'raw_sections': ['query_graph',
  'knowledge_graph',
  'results',
  'auxiliary_graphs']}

In [5]:
import json
import datetime
with open(f'TCT_path_finder_result__{paths.resolved_nodes["start"].input_value}__{paths.resolved_nodes["end"].input_value}.json', 'w') as f:
    json.dump(paths.raw, f, indent=4)


In [ ]:
# A step by step pathfinder query

In [3]:
import sys
import os
sys.path.append('../TCT/')
import TCT
from TCT import translator_kpinfo
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_query
from TCT import TCT_pathfinder

import requests

In [4]:
APInames, metaKG, Translator_KP_info= translator_metakg.load_translator_resources()

All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

    # generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

Skipping server without x-maturity: {'url': '/sipr'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}


In [5]:
# This is an example of selecting a list of APIs for the neighborhood finder. The user can modify this list to include the APIs they want to use. The APIs in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. The user can also modify the list of predicates to use for finding the neighborhood. The predicates in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. 
# The user can also modify the list of categories to use for finding the neighborhood. 
# The categories in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph.
# if selected_APIlist is empty, use all APIs in APInames
selected_APIlist = ['Retriever',
                    'Clinical Trials KP - TRAPI 1.5.0',
                    'Drug Approvals KP - TRAPI 1.5.0',
                    'Genetics Data Provider for NCATS Biomedical Translator Reasoners',
                    'Microbiome KP - TRAPI 1.5.0',
                    'MolePro',
                    'COHD TRAPI',
                    'RTX KG2 - TRAPI 1.5.0',
                    'Text Mined Cooccurrence API',
                    'CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0',
                    'CATRAX Pharmacogenomics KP - TRAPI 1.5.0',
                    ]

# add Automat API to the selected API list if it is not already in the list
for api in APInames:
    if 'Automat' in api and api not in selected_APIlist:
        selected_APIlist.append(api)

#selected_APIlist = ['Retriever']

# select a list of APIs to use and a list of predicates to use
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}

selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
#print(select_APIs)
print(selected_metaKG.shape)

All_predicates = list(set(selected_metaKG['Predicate']))
All_categories = list((set(list(set(selected_metaKG['Subject']))+list(set(selected_metaKG['Object'])))))
API_withMetaKG = list(set(selected_metaKG['API']))
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(selected_metaKG[selected_metaKG['API'] == api]['Predicate']))

(19907, 5)


In [6]:
#name_resolver.lookup('miR-155', only_taxa='NCBITaxon:9606')
subject_name = 'Revumenib'
subject_node = name_resolver.lookup(subject_name).curie
subject_category = name_resolver.lookup(subject_name).types
subject_category = ["biolink:Drug", "biolink:SmallMolecule", "biolink:ChemicalSubstance"]

object_name = 'acute myeloid leukemia'
object_node = name_resolver.lookup(object_name, biolink_category='biolink:Disease').curie
object_category = name_resolver.lookup(object_name, biolink_category='biolink:Disease').types
object_category = ["biolink:Disease"]

intermediate_categories = [ 'biolink:Gene','biolink:Protein']

#name_resolver.lookup('Myelodysplastic syndrome', return_top_response=False, biolink_category='biolink:Disease')

In [7]:
print(subject_node , object_node)

PUBCHEM.COMPOUND:132212657 MONDO:0018874


In [8]:
from TCT import query_TCT_pathfinder

result = query_TCT_pathfinder(start=subject_node,  # e.g. IFNG
                    end=object_node,  # e.g. disease node
                    intermediate_categories=intermediate_categories,
                    api_names=select_APIs,
                    meta_kg=selected_metaKG,
                    api_predicates=API_predicates)


RTX KG2 - TRAPI 1.5.0: Success!
MolePro: Success!
CATRAX Pharmacogenomics KP - TRAPI 1.5.0: Success!
CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0: Success!
Clinical Trials KP - TRAPI 1.5.0: Success!
RTX KG2 - TRAPI 1.5.0: Success!


MolePro: Success!


CATRAX Pharmacogenomics KP - TRAPI 1.5.0: Success!


CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0: Success!
Clinical Trials KP - TRAPI 1.5.0: Success!


RTX KG2 - TRAPI 1.5.0: Success!


In [9]:
# pathfinder already returns parsed results inside a FinderResult
import json
with open(f'TCT_path_finder_result__{subject_node.replace(":", "_")}__{object_node.replace(":", "_")}.json', 'w') as f:
    json.dump(result.to_dict(), f, indent=4)


In [10]:
# visulize the paths in visulize_path_finder_results.html

In [11]:
intermediate_categories = ['biolink:Gene']

In [12]:
# query arax pathfinder with constraints and write response to a json file
import time
start_time = time.time()
result_arax = TCT_pathfinder.query_arax_pathfinder_with_constraints(subject_node, 
                                                                    subject_category,
                                                                    object_node, 
                                                                    object_category, 
                                                                    constraints=intermediate_categories
                                                                
                                                                    )
end_time = time.time()
arax_execution_time = end_time - start_time
if result_arax.json()['message'].get('auxiliary_graphs') is not None:
    Number_of_paths_ARAX_constrained = len(result_arax.json()['message']['auxiliary_graphs'])
else:    
    Number_of_paths_ARAX_constrained = 0
print(f"ARAX execution time with constraints: {arax_execution_time} seconds")
print(f"Number of paths ARAX with constraints: {Number_of_paths_ARAX_constrained}")
import json
with open('arax_pathfinder_response_'+subject_name.replace(":", "_")+'_'+object_name.replace(":", "_")+'with_constraints.json', 'w') as f:
    json.dump(result_arax.json()['message'], f, indent=4)

ARAX execution time with constraints: 28.263652324676514 seconds
Number of paths ARAX with constraints: 500


In [13]:
# run aragorn pathfinder without constraints and write response to a json file
import time
start_time = time.time()

# Note: for ARAGORN or ARAX pathfinder pipeline, it is only allowed to have only one intermediate category in the constraints list. If there are multiple intermediate categories, the query will return an error. Therefore, we will only use one intermediate category in  the constraints list. 

aragorn_response = TCT_pathfinder.query_aragorn_pathfinder_with_constraints(subject_node, 
                                                           subject_category, 
                                                           object_node, 
                                                           object_category, 
                                                           constraints=intermediate_categories)
# write response to a json file
import json
if 'message' in aragorn_response.json():
    with open('aragorn_pathfinder_response_'+subject_name.replace(":", "_")+'_'+object_name.replace(":", "_")+'constrained.json', 'w') as f:
        json.dump(aragorn_response.json()['message'], f, indent=4)
end_time = time.time()
aragorn_execution_time = end_time - start_time
if 'auxiliary_graphs' in aragorn_response.json()['message']:
    Number_of_paths_aragorn = len(aragorn_response.json()['message']['auxiliary_graphs'])
else:
    Number_of_paths_aragorn = 0
    print("No paths found in Aragorn response.")
print(f"Aragorn execution time: {aragorn_execution_time} seconds")
print(f"Number of paths Aragorn: {Number_of_paths_aragorn}")

Aragorn execution time: 2.2056174278259277 seconds
Number of paths Aragorn: 29


In [14]:
# run aragorn pathfinder without constraints and write response to a json file
start_time = time.time()
aragorn_response = TCT_pathfinder.query_aragorn_pathfinder(subject_node, 
                                                           subject_category, 
                                                           object_node, 
                                                           object_category)
# write response to a json file
import json
if 'message' in aragorn_response.json():
    with open('aragorn_pathfinder_response_'+subject_name.replace(":", "_")+'_'+object_name.replace(":", "_")+'.json', 'w') as f:
        json.dump(aragorn_response.json()['message'], f, indent=4)
end_time = time.time()
aragorn_execution_time = end_time - start_time
if aragorn_response.json()['message'].get('auxiliary_graphs') is not None:
    Number_of_paths_aragorn = len(aragorn_response.json()['message']['auxiliary_graphs'])
else:
    Number_of_paths_aragorn = 0
    print("No paths found in Aragorn response.")
print(f"Aragorn execution time: {aragorn_execution_time} seconds")
print(f"Number of paths Aragorn: {Number_of_paths_aragorn}")

Aragorn execution time: 2.178687810897827 seconds
Number of paths Aragorn: 29


In [15]:
arax_response = TCT_pathfinder.query_arax_pathfinder(subject_node, 'biolink:SmallMolecule', object_node, 'biolink:Disease')
# write response to a json file
import json
with open('arax_pathfinder_response_'+subject_node.replace(":", "_")+'_'+object_node.replace(":", "_")+'.json', 'w') as f:
    json.dump(arax_response.json()['message'], f, indent=4)
arax_execution_time = end_time - start_time
if arax_response.json()['message'].get('auxiliary_graphs') is not None:
    Number_of_paths_ARAX = len(arax_response.json()['message']['auxiliary_graphs'])
else:    
    Number_of_paths_ARAX = 0
print(f"ARAX execution time: {arax_execution_time} seconds")
print(f"Number of paths ARAX: {Number_of_paths_ARAX}")


ARAX execution time: 2.178687810897827 seconds
Number of paths ARAX: 500


# link to the UI
https://ui.ci.transltr.io/

# link to TCT results visulization
./visulize_path_finder_results.html
